# 01 — NBA API exploration

Roadmap step 1. Goal: find out what the endpoints **actually** return before any
pipeline code gets written, so the staging tables are shaped by real payloads
rather than guesses.

Sample date: **2026-01-15** (mid-season, 9 games). Today is the offseason, so
exploration has to point at an in-season date. This is not backfill — nothing
here writes to the database.

## Headline findings

1. **`boxscoretraditionalv2` is dead.** It still returns the right *columns* but
   **zero rows**: *"Data is no longer being published for BoxScoreTraditionalV2
   as of the 2025-26 NBA season."* Use **`BoxScoreTraditionalV3`**. `CLAUDE.md`
   named v2; that reference is now corrected.
2. **V3 renames everything to camelCase** (`turnovers` not `TO`, `points` not
   `PTS`, `personId` not `PLAYER_ID`). Our staging layer has to map these.
3. **`minutes` is a `"MM:SS"` string**, not a number. Must be parsed to decimal
   minutes before any per-minute or rating math.
4. **DNP rows are present in the box score** with an empty `minutes` and a
   populated `comment` (`"DNP - Coach's Decision"`, `"DND - Injury/Illness"`).
   6 of 26 rows in the sample game. These need an explicit decision, not a
   silent `0`.
5. **`position` is only set for starters** — blank for the other 16 of 26 rows.
   So `dim_players.position` cannot come from the box score alone.
6. **`MATCHUP` is not reliably row-relative** — see the section below. Parse it
   structurally instead.
7. **`playergamelog` is one call per player per season** — the wrong shape for a
   daily job. The box score already carries every player who dressed.

In [ ]:
from nba_api.stats.endpoints import (
    leaguegamefinder,
    boxscoretraditionalv3,
    playergamelog,
)
import pandas as pd
import time

pd.set_option("display.width", 240)
pd.set_option("display.max_columns", 60)

SAMPLE_DATE = "01/15/2026"   # nba_api wants MM/DD/YYYY
REQUEST_DELAY = 1.0          # be polite to stats.nba.com

## 1. `leaguegamefinder` — which games happened on a date

This is the **game-discovery** step: given a date, which games were played?
It returns *team*-level rows, **two per game** (one for each side), so 9 games
came back as 18 rows. It doubles as the source for `fact_team_game_stats`.

In [ ]:
games = leaguegamefinder.LeagueGameFinder(
    date_from_nullable=SAMPLE_DATE,
    date_to_nullable=SAMPLE_DATE,
    league_id_nullable="00",   # "00" = NBA (excludes G-League / WNBA)
    timeout=60,
).get_data_frames()[0]

print(f"{len(games)} rows / {games.GAME_ID.nunique()} games")
games[["GAME_ID", "TEAM_ID", "TEAM_ABBREVIATION", "MATCHUP", "WL", "PTS"]]

### ID formats worth knowing

- `GAME_ID` = `"0022500578"` — `002` season type (regular season) + `25` season
  start year + 5-digit game sequence. It's a **zero-padded string**; storing it
  as an integer would destroy the leading zeros. Keep it `TEXT`.
- `SEASON_ID` = `"22025"` — leading `2` = regular season, `2025` = season start
  year. Playoffs would be `4`-prefixed.
- `TEAM_ID` = a stable 10-digit NBA id (`1610612753`). Safe as a `BIGINT` PK.

In [ ]:
print("GAME_ID dtype:", games.GAME_ID.dtype)
print("sample:", games.GAME_ID.iloc[0], "| SEASON_ID:", games.SEASON_ID.iloc[0])
print("\nrows per game (expect exactly 2):")
print(games.GAME_ID.value_counts().to_string())

### The `MATCHUP` trap — deriving home vs. away

`dim_games` needs `home_team_id` / `away_team_id`, and `MATCHUP` is the only
place that information lives. The obvious rule — *"if my row's `MATCHUP` says
`vs.` I'm home"* — **is wrong**.

In the sample, game `0022500578` has **both** rows reading `"MEM @ ORL"`: the
ORL row does *not* lead with `ORL`. Every other game that day was row-relative
(`DET vs. PHX` / `PHX @ DET`), so this fails on roughly 1 game in 9 — often
enough to corrupt home/away splits, rarely enough to slip through a spot check.

The robust rule reads the string **structurally**, ignoring which row it is on:

- `"AWAY @ HOME"` — the `@` form always puts the home team second
- `"HOME vs. AWAY"` — the `vs.` form always puts the home team first

Both forms agree for every row of a given game, so either row yields the same
answer.

In [ ]:
def parse_matchup(matchup: str) -> tuple[str, str]:
    """Return (home_abbrev, away_abbrev) from a MATCHUP string.

    Structural, not row-relative: 'X @ Y' -> home Y, away X;
    'X vs. Y' -> home X, away Y.
    """
    if " vs. " in matchup:
        home, away = matchup.split(" vs. ")
    elif " @ " in matchup:
        away, home = matchup.split(" @ ")
    else:
        raise ValueError(f"unrecognised MATCHUP format: {matchup!r}")
    return home.strip(), away.strip()


check = games.assign(
    parsed=games.MATCHUP.map(parse_matchup),
).assign(
    home=lambda d: d.parsed.str[0],
    away=lambda d: d.parsed.str[1],
)

# Both rows of every game must agree on who was home.
agreement = check.groupby("GAME_ID")[["home", "away"]].nunique()
print("games where the two rows disagree:", int((agreement > 1).sum().sum()))

check[["GAME_ID", "TEAM_ABBREVIATION", "MATCHUP", "home", "away"]]

### The naive rule, for contrast

Confirming the failure is real rather than theoretical — this is the version we
are deliberately *not* shipping.

In [ ]:
naive_home = games.MATCHUP.str.contains(" vs. ")
naive = games.assign(naive_is_home=naive_home)

# Correct answer: the row's own team equals the structurally-parsed home team.
truth = check.TEAM_ABBREVIATION == check.home
mismatch = naive[naive_home.values != truth.values]

print(f"{len(mismatch)} of {len(games)} rows misclassified by the naive rule:")
mismatch[["GAME_ID", "TEAM_ABBREVIATION", "MATCHUP", "naive_is_home"]]

## 2. `boxscoretraditionalv2` — deprecated, returns nothing

Kept here as evidence, because the brief named this endpoint. It does not raise;
it returns correctly-shaped **empty** frames. A pipeline built on it would have
run "successfully" every night and quietly loaded zero rows — the worst kind of
failure. Left commented out so the notebook doesn't waste a request.

In [ ]:
# from nba_api.stats.endpoints import boxscoretraditionalv2
#
# dead = boxscoretraditionalv2.BoxScoreTraditionalV2(
#     game_id="0022500578", timeout=60
# ).get_data_frames()[0]
#
# DeprecationWarning: BoxScoreTraditionalV2 is deprecated ... Data is no longer
# being published for BoxScoreTraditionalV2 as of the 2025-26 NBA season.
#
# len(dead) -> 0   (with all 29 expected columns present)

## 3. `boxscoretraditionalv3` — the real player-level source

One call per `game_id`. Returns three frames:

| frame | rows | what |
|---|---|---|
| 0 | 26 | **player** box score — feeds `fact_player_game_stats` |
| 1 | 4 | starters/bench subtotals — not needed |
| 2 | 2 | **team** totals — cross-check for `fact_team_game_stats` |

Frame 0 is the one that matters.

In [ ]:
box = boxscoretraditionalv3.BoxScoreTraditionalV3(
    game_id="0022500578", timeout=60
)
players, starters_bench, teams = box.get_data_frames()

print("frame row counts:", len(players), len(starters_bench), len(teams))
print("\nplayer columns:")
print(list(players.columns))
players.head(3)

### V3 → our schema column mapping

The names our `fact_player_game_stats` spec uses vs. what V3 actually calls
them. Note `turnovers` (v3) was `TO` (v2) but our column is `turnovers`, and
`pf` maps from `foulsPersonal`.

In [ ]:
COLUMN_MAP = {
    "gameId": "game_id",
    "personId": "player_id",
    "teamId": "team_id",
    "minutes": "min",             # "MM:SS" string -> parse below
    "points": "pts",
    "reboundsTotal": "reb",
    "reboundsOffensive": "oreb",
    "reboundsDefensive": "dreb",
    "assists": "ast",
    "steals": "stl",
    "blocks": "blk",
    "fieldGoalsMade": "fgm",
    "fieldGoalsAttempted": "fga",
    "threePointersMade": "fg3m",
    "threePointersAttempted": "fg3a",
    "freeThrowsMade": "ftm",
    "freeThrowsAttempted": "fta",
    "turnovers": "turnovers",
    "foulsPersonal": "pf",
    "plusMinusPoints": "plus_minus",
}

missing = set(COLUMN_MAP) - set(players.columns)
print("mapped columns missing from the payload:", missing or "none")
players.rename(columns=COLUMN_MAP)[list(COLUMN_MAP.values())].head(3)

### `minutes` is `"MM:SS"`, not a number

`"26:41"` is 26.68 minutes, not 26.41. Anything rate-based (pace, ratings,
per-36) is wrong if this is parsed as a float. Convert at load time.

In [ ]:
print("raw samples:", players.minutes.unique()[:6].tolist())


def parse_minutes(value) -> float | None:
    """'26:41' -> 26.683. Empty/blank (a DNP) -> None."""
    if value is None or str(value).strip() == "":
        return None
    mins, _, secs = str(value).partition(":")
    return round(int(mins) + int(secs or 0) / 60, 3)


players.assign(min_decimal=players.minutes.map(parse_minutes))[
    ["familyName", "minutes", "min_decimal"]
].head(6)

### DNP rows are in the payload

6 of 26 rows are players who did not play: blank `minutes`, zeroed stats, and a
populated `comment`. **Decision for the ingestion layer:** keep them in staging
(they are a true fact about the game — the API said this player was rostered and
did not play), but the transform into `fact_player_game_stats` filters to rows
that actually played. Otherwise every rolling average gets dragged toward zero
by DNPs.

In [ ]:
dnp = players[players.comment.str.strip() != ""]
print(f"{len(dnp)} of {len(players)} rows are DNP/DND")
dnp[["firstName", "familyName", "comment", "minutes", "points"]]

In [ ]:
played = players[players.comment.str.strip() == ""]
print(f"{len(played)} players actually played")
print("\nnulls among players who played:")
print(played[["minutes", "points", "plusMinusPoints"]].isna().sum().to_string())

### `position` only exists for starters

10 of 26 rows (5 per team) carry a position; the bench is blank. So
`dim_players.position` **cannot** be populated from box scores alone — a player
who never starts would never get one. Options: source it from
`commonplayerinfo` / the static player list, or carry "last known starting
position" and accept the nulls. Flagging for step 3, not solving it here.

In [ ]:
blank = (players.position.str.strip() == "").sum()
print(f"blank position: {blank} of {len(players)}")
players.groupby(players.position.replace("", "(blank)")).size()

### Team totals (frame 2) — the cross-check

Frame 2 gives per-team totals for the same game. Two uses: it feeds
`fact_team_game_stats`, and summing the player rows should reproduce it — a
free **data-quality check** for step 3.

In [ ]:
player_pts = players.groupby("teamTricode").points.sum()
team_pts = teams.set_index("teamTricode").points

comparison = pd.DataFrame({"sum_of_players": player_pts, "team_total": team_pts})
comparison["match"] = comparison.sum_of_players == comparison.team_total
comparison

## 4. `playergamelog` — wrong shape for daily ingestion

One call per **player per season**, returning that player's whole season. To
cover one night's games we'd need ~250 calls instead of the 9 that
`boxscoretraditionalv3` needs — and at a ~1s delay that is minutes of requests
for data we already have.

**Conclusion: not used by the pipeline.** The box score carries everything, and
rolling averages are the transform layer's job (that's the point of the
project). Noted here so the decision isn't re-litigated later.

In [ ]:
time.sleep(REQUEST_DELAY)

log = playergamelog.PlayerGameLog(
    player_id=1631094,   # Paolo Banchero
    season="2025-26",
    timeout=60,
).get_data_frames()[0]

print(f"{len(log)} rows for ONE player across the whole season")
print("note: snake-free UPPERCASE columns, and GAME_DATE is 'Apr 12, 2026'")
log.head(3)

## What this means for step 2

- Ingest with **two** endpoints: `leaguegamefinder` (one call per date, gives the
  game list + team stats) then `boxscoretraditionalv3` (one call per game).
  That's `1 + N` calls per night, ~10 for a typical slate.
- Store `game_id` as `TEXT` — leading zeros are load-bearing.
- Parse `minutes` from `"MM:SS"` at load; parse home/away from `MATCHUP`
  structurally.
- Land DNP rows in staging, filter them in the transform.
- `dim_players.position` needs a source that isn't the box score.
- Keep the `~1s` delay and add retry-with-backoff, per `CLAUDE.md`.